# Lab 47 — DMRG para la cadena de Heisenberg

Implementamos DMRG (Density Matrix Renormalization Group) desde cero con numpy/scipy para encontrar el estado fundamental de la cadena de Heisenberg antiferromagnética en 1D, y comparamos con VQE de Qiskit.

**Hamiltoniano:**
$$H = J\sum_{k=0}^{n-2}(X_k X_{k+1} + Y_k Y_{k+1} + Z_k Z_{k+1}), \quad J = 1$$

**Contenido:**
- Parte A: Construcción del Hamiltoniano y diagonalización exacta (referencia)
- Parte B: DMRG infinite-size algorithm (iDMRG)
- Parte C: DMRG finite-size sweeps
- Parte D: Propiedades del estado fundamental (entropía, correlaciones)
- Parte E: Comparativa DMRG vs VQE vs ED

**Referencia:** Módulo 42 — Tensor Networks y DMRG

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.linalg import eigh, svd
from scipy.sparse import kron, eye as speye, csr_matrix
from scipy.sparse.linalg import eigsh
from scipy.optimize import minimize

print('Entorno listo para DMRG')

## Parte A: Hamiltoniano de Heisenberg y Exact Diagonalization

Construimos $H$ en la base computacional y obtenemos la energía exacta como referencia para DMRG.

In [ ]:
# Operadores de Pauli
I2 = np.eye(2)
Sx = np.array([[0., 1.], [1., 0.]]) / 2
Sy = np.array([[0., -1j], [1j, 0.]]) / 2
Sz = np.array([[1., 0.], [0., -1.]]) / 2


def heisenberg_hamiltonian(n: int, J: float = 1.0, periodic: bool = False) -> np.ndarray:
    """Hamiltoniano de Heisenberg H = J Σ(XX+YY+ZZ) en cadena 1D de n sitios."""
    dim = 2**n
    H = np.zeros((dim, dim), dtype=complex)

    def embed(op1, op2, k, n):
        """Embebe op1 en sitio k y op2 en sitio k+1 en n qubits."""
        ops = [I2] * n
        ops[k] = op1
        ops[k + 1] = op2
        result = ops[0]
        for op in ops[1:]:
            result = np.kron(result, op)
        return result

    bonds = list(range(n - 1))
    if periodic:
        bonds.append(n - 1)  # enlace (n-1, 0)

    for k in bonds:
        k1 = k % n
        k2 = (k + 1) % n
        if k < n - 1:
            H += J * (embed(Sx, Sx, k1, n) * 4 +
                      embed(Sy, Sy, k1, n) * 4 +
                      embed(Sz, Sz, k1, n) * 4)
        # Nota: S = sigma/2, así SxSx = (XX)/4
    return np.real(H)


def exact_ground_energy(n: int, J: float = 1.0) -> tuple:
    """Energía y vector del estado fundamental por ED."""
    H = heisenberg_hamiltonian(n, J)
    evals, evecs = eigh(H)
    return float(evals[0]), evecs[:, 0]


# Energía exacta de referencia (Bethe ansatz: E0/n → -ln2 + 1/4 ≈ -0.4431 para n→∞)
E_bethe_per_site = -(np.log(2) - 0.25)  # límite termodinámico exacto

print('Exact Diagonalization — cadena de Heisenberg antiferromagnética:')
print(f'  (Referencia Bethe ansatz n→∞: E₀/n = {E_bethe_per_site:.6f})')
print()

ed_results = {}
for n in [4, 6, 8, 10]:
    E0, psi0 = exact_ground_energy(n)
    ed_results[n] = (E0, psi0)
    print(f'  n={n}: E₀={E0:.6f}  E₀/n={E0/n:.6f}')

## Parte B: iDMRG — algoritmo infinite-size

El iDMRG crece el sistema de 2 sitios hasta alcanzar el tamaño deseado, manteniendo la bond dimension fija. En cada paso:
1. Expandir el bloque izquierdo y derecho añadiendo un sitio.
2. Resolver el eigenproblema en el super-bloque de 4 sitios.
3. Truncar los estados del bloque a los $\chi$ más relevantes (mayor peso en la matriz densidad).

In [ ]:
class DMRGBlock:
    """Bloque del sistema para DMRG."""

    def __init__(self, length: int, basis_size: int,
                 H_block: np.ndarray, conn_Sp: np.ndarray, conn_Sm: np.ndarray, conn_Sz: np.ndarray):
        self.length = length
        self.basis_size = basis_size
        self.H = H_block          # Hamiltoniano del bloque
        self.Sp = conn_Sp         # S+ en el sitio de conexión
        self.Sm = conn_Sm         # S- en el sitio de conexión
        self.Sz = conn_Sz         # Sz en el sitio de conexión


def initial_block() -> DMRGBlock:
    """Bloque inicial de 1 sitio."""
    Sp = np.array([[0., 1.], [0., 0.]])
    Sm = np.array([[0., 0.], [1., 0.]])
    Sz = np.array([[0.5, 0.], [0., -0.5]])
    H = np.zeros((2, 2))
    return DMRGBlock(1, 2, H, Sp, Sm, Sz)


def enlarge_block(block: DMRGBlock, J: float = 1.0) -> DMRGBlock:
    """Añade un sitio al bloque (crecimiento hacia la derecha)."""
    d = 2  # dimensión física
    m = block.basis_size

    Sp_site = np.array([[0., 1.], [0., 0.]])
    Sm_site = np.array([[0., 0.], [1., 0.]])
    Sz_site = np.array([[0.5, 0.], [0., -0.5]])

    # Hamiltoniano ampliado: H_block ⊗ I + interacción + I ⊗ H_site
    H_new = np.kron(block.H, np.eye(d)) + J * (
        0.5 * np.kron(block.Sp, Sm_site) +
        0.5 * np.kron(block.Sm, Sp_site) +
        np.kron(block.Sz, Sz_site)
    )

    # Operadores de conexión en el nuevo sitio
    Sp_new = np.kron(np.eye(m), Sp_site)
    Sm_new = np.kron(np.eye(m), Sm_site)
    Sz_new = np.kron(np.eye(m), Sz_site)

    return DMRGBlock(block.length + 1, m * d, H_new, Sp_new, Sm_new, Sz_new)


def truncate_block(block: DMRGBlock, psi_gs: np.ndarray, m_left: int, m_right: int, chi: int) -> DMRGBlock:
    """Trunca el bloque usando la matriz densidad del estado fundamental."""
    # Reshape del estado fundamental al super-bloque
    psi_mat = psi_gs.reshape(m_left, m_right)
    # Matriz densidad reducida del bloque izquierdo
    rho = psi_mat @ psi_mat.conj().T
    # Eigendescomposición de la matriz densidad
    evals, evecs = eigh(rho)
    # Ordenar por eigenvalor descendente y tomar los chi primeros
    idx = np.argsort(evals)[::-1]
    chi_actual = min(chi, len(evals))
    T = evecs[:, idx[:chi_actual]]  # transformación de truncamiento

    # Truncar Hamiltoniano y operadores
    H_trunc = T.conj().T @ block.H @ T
    Sp_trunc = T.conj().T @ block.Sp @ T
    Sm_trunc = T.conj().T @ block.Sm @ T
    Sz_trunc = T.conj().T @ block.Sz @ T

    return DMRGBlock(block.length, chi_actual, H_trunc, Sp_trunc, Sm_trunc, Sz_trunc)


def idmrg(n_sites: int, chi: int = 20, J: float = 1.0) -> tuple:
    """iDMRG: crece el sistema hasta n_sites sitios. Devuelve (energía/sitio, bloques)."""
    block_L = initial_block()
    block_R = initial_block()
    energies = []

    for step in range(n_sites // 2 - 1):
        # Ampliar ambos bloques
        block_L = enlarge_block(block_L, J)
        block_R = enlarge_block(block_R, J)

        m_L = block_L.basis_size
        m_R = block_R.basis_size
        dim_super = m_L * m_R

        # Hamiltoniano del super-bloque
        H_super = (np.kron(block_L.H, np.eye(m_R)) +
                   np.kron(np.eye(m_L), block_R.H) +
                   J * (0.5 * np.kron(block_L.Sp, block_R.Sm) +
                        0.5 * np.kron(block_L.Sm, block_R.Sp) +
                        np.kron(block_L.Sz, block_R.Sz)))

        # Diagonalizar (Lanczos para sistemas grandes)
        if dim_super <= 200:
            evals, evecs = eigh(H_super)
            E0 = evals[0]
            psi_gs = evecs[:, 0]
        else:
            evals, evecs = eigsh(csr_matrix(H_super), k=1, which='SA')
            E0 = float(evals[0])
            psi_gs = evecs[:, 0]

        n_total = 2 * (step + 2)
        E_per_site = E0 / n_total
        energies.append((n_total, E0, E_per_site))

        # Truncar bloques
        block_L = truncate_block(block_L, psi_gs, m_L, m_R, chi)
        block_R = truncate_block(block_R, psi_gs.reshape(m_L, m_R).T.flatten(), m_R, m_L, chi)

    return energies, block_L, block_R


# Ejecutar iDMRG
chi = 32
print(f'iDMRG (χ={chi}) — crecimiento del sistema:')
print(f'  (Referencia Bethe ansatz: E₀/n = {E_bethe_per_site:.6f})')
print()

energies_idmrg, block_L_final, block_R_final = idmrg(n_sites=30, chi=chi)

for n_tot, E0, E_per in energies_idmrg[-5:]:
    err = abs(E_per - E_bethe_per_site)
    print(f'  n={n_tot:3d}: E₀/n = {E_per:.6f}  error = {err:.2e}')

# Comparar con ED para n=10
n_compare = 10
E0_ed = ed_results[n_compare][0] / n_compare if n_compare in ed_results else None
E_idmrg_n10 = next((E for n, E0, E in energies_idmrg if n == n_compare), None)
if E0_ed and E_idmrg_n10:
    print(f'\nComparativa n={n_compare}:')
    print(f'  ED:    E₀/n = {E0_ed:.6f}')
    print(f'  iDMRG: E₀/n = {E_idmrg_n10:.6f}  error = {abs(E_idmrg_n10 - E0_ed):.2e}')

## Parte C: Convergencia y escalado con χ

In [ ]:
# Energía vs χ para sistema fijo n=20
chi_vals = [4, 8, 16, 32, 64]
E_vs_chi = []

for chi_test in chi_vals:
    ens, _, _ = idmrg(n_sites=20, chi=chi_test)
    # Tomar energía del sistema de n=20
    E_n20 = next((E for n, E0, E in ens if n == 20), ens[-1][2])
    E_vs_chi.append(E_n20)
    print(f'  χ={chi_test:3d}: E₀/n = {E_n20:.8f}')

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Convergencia con χ
axes[0].semilogx(chi_vals, E_vs_chi, 'b-o', lw=2, ms=8)
axes[0].axhline(E_bethe_per_site, color='red', ls='--', lw=2,
                label=f'Bethe ansatz: {E_bethe_per_site:.6f}')
axes[0].set_xlabel('Bond dimension χ', fontsize=11)
axes[0].set_ylabel('E₀/n', fontsize=11)
axes[0].set_title('Convergencia de iDMRG con χ (n=20)', fontsize=12)
axes[0].legend(fontsize=9)
axes[0].grid(True, alpha=0.3)

# Escalado en tamaño del sistema
n_vals = [n for n, _, _ in energies_idmrg]
E_vals = [E for _, _, E in energies_idmrg]

axes[1].plot(n_vals, E_vals, 'g-o', lw=2, ms=5, label='iDMRG')
axes[1].axhline(E_bethe_per_site, color='red', ls='--', lw=2, label='Bethe ansatz (n→∞)')

# ED para comparar
ed_ns = sorted(ed_results.keys())
ed_Es = [ed_results[n][0] / n for n in ed_ns]
axes[1].plot(ed_ns, ed_Es, 'rs', ms=8, label='ED exacta', zorder=5)

axes[1].set_xlabel('Número de sitios n', fontsize=11)
axes[1].set_ylabel('E₀/n', fontsize=11)
axes[1].set_title('Energía vs tamaño del sistema', fontsize=12)
axes[1].legend(fontsize=9)
axes[1].grid(True, alpha=0.3)

plt.suptitle('iDMRG — Cadena de Heisenberg antiferromagnética', fontsize=13)
plt.tight_layout()
plt.show()

## Parte D: Propiedades del estado fundamental

Calculamos la función de correlación $C(r) = \langle S^z_0 S^z_r \rangle$ y la entropía de entrelazamiento del estado fundamental de ED.

In [ ]:
def spin_correlation(psi: np.ndarray, i: int, j: int, n: int) -> float:
    """⟨Sz_i Sz_j⟩ para el estado psi de n qubits."""
    rho = np.outer(psi, psi.conj())
    dim = 2**n
    result = 0.0
    for idx in range(dim):
        bi = (idx >> (n-1-i)) & 1
        bj = (idx >> (n-1-j)) & 1
        sz_i = 0.5 - bi
        sz_j = 0.5 - bj
        result += np.real(rho[idx, idx]) * sz_i * sz_j
    return result


def entanglement_profile(psi: np.ndarray, n: int) -> list:
    """Entropía de entrelazamiento para cada corte bipartito."""
    entropies = []
    for cut in range(1, n):
        M = psi.reshape(2**cut, 2**(n-cut))
        _, S, _ = svd(M, full_matrices=False)
        lam2 = S**2; lam2 = lam2[lam2 > 1e-14]
        S_ent = float(-np.sum(lam2 * np.log2(lam2)))
        entropies.append(S_ent)
    return entropies


n_prop = 10
E0_prop, psi0_prop = ed_results[n_prop]

# Función de correlación desde el sitio central
center = n_prop // 2
r_vals = list(range(1, n_prop - center))
C_r = [spin_correlation(psi0_prop, center, center + r, n_prop) for r in r_vals]

# Perfil de entropía de entrelazamiento
entropies = entanglement_profile(psi0_prop, n_prop)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Correlación
axes[0].plot(r_vals, C_r, 'b-o', lw=2, ms=7)
# Ajuste: ley de potencias C(r) ~ (-1)^r / r (Heisenberg 1D antiferro)
r_arr = np.array(r_vals)
C_fit = [(-1)**r * 0.25 / r for r in r_vals]  # comportamiento asintótico
axes[0].plot(r_vals, C_fit, 'r--', lw=1.5, alpha=0.7, label='(-1)ʳ × 0.25/r (asintótico)')
axes[0].axhline(0, color='gray', ls=':', lw=1)
axes[0].set_xlabel('Separación r', fontsize=11)
axes[0].set_ylabel('⟨Sz₀ Szᵣ⟩', fontsize=11)
axes[0].set_title(f'Correlación de espín — Heisenberg n={n_prop}', fontsize=12)
axes[0].legend(fontsize=9)
axes[0].grid(True, alpha=0.3)

# Perfil de entropía (area law logarítmica para sistema crítico)
cuts = list(range(1, n_prop))
axes[1].plot(cuts, entropies, 'g-s', lw=2, ms=7, label='S(corte) numérica')

# Ajuste CFT: S = (c/3) log(n/π · sin(πl/n)) + cte
c = 1.0  # cadena de Heisenberg tiene c=1
S_cft = [(c / 3) * np.log(n_prop / np.pi * np.sin(np.pi * l / n_prop)) + 0.73 for l in cuts]
axes[1].plot(cuts, S_cft, 'r--', lw=1.5, alpha=0.8, label=f'CFT (c={c})')

axes[1].set_xlabel('Corte bipartito l', fontsize=11)
axes[1].set_ylabel('S(l) (ebits)', fontsize=11)
axes[1].set_title(f'Entropía de entrelazamiento — ley de área logarítmica', fontsize=12)
axes[1].legend(fontsize=9)
axes[1].grid(True, alpha=0.3)

plt.suptitle(f'Propiedades del estado fundamental — Heisenberg n={n_prop}', fontsize=13)
plt.tight_layout()
plt.show()

print('La cadena de Heisenberg es un sistema crítico (sin gap, c=1):')
print('  C(r) ~ (-1)ʳ/r  (decaimiento algebraico con alternancia antiferromagnética)')
print('  S(l) ~ (1/3)log(l) + cte  (ley de área logarítmica, CFT con c=1)')

## Parte E: Comparativa DMRG vs VQE vs ED

In [ ]:
from qiskit import QuantumCircuit
from qiskit.circuit import ParameterVector
from qiskit.quantum_info import SparsePauliOp
from qiskit.primitives import StatevectorEstimator


def heisenberg_hamiltonian_qiskit(n: int, J: float = 1.0) -> SparsePauliOp:
    """Hamiltoniano de Heisenberg como SparsePauliOp para Qiskit."""
    terms = []
    for k in range(n - 1):
        for gate in ['XX', 'YY', 'ZZ']:
            pauli = ['I'] * n
            pauli[k] = gate[0]
            pauli[k + 1] = gate[1]
            terms.append((''.join(reversed(pauli)), J / 4))
    return SparsePauliOp.from_list(terms, num_qubits=n).simplify()


def vqe_heisenberg(n: int, n_layers: int = 3) -> float:
    """VQE para la cadena de Heisenberg de n qubits."""
    H = heisenberg_hamiltonian_qiskit(n)
    params = ParameterVector('θ', n * n_layers * 2)

    qc = QuantumCircuit(n)
    # Estado inicial: Néel antiferro (aproximación al estado fundamental)
    for k in range(0, n, 2):
        qc.x(k)

    idx = 0
    for _ in range(n_layers):
        for q in range(n):
            qc.ry(params[idx], q)
            qc.rz(params[idx + 1], q)
            idx += 2
        for q in range(0, n - 1, 2):
            qc.cx(q, q + 1)
        for q in range(1, n - 1, 2):
            qc.cx(q, q + 1)

    estimator = StatevectorEstimator()
    energy_hist = []

    def cost(p):
        job = estimator.run([(qc, H, [p])])
        E = float(job.result()[0].data.evs)
        energy_hist.append(E)
        return E

    np.random.seed(0)
    theta0 = np.random.uniform(-0.5, 0.5, qc.num_parameters)
    result = minimize(cost, theta0, method='COBYLA',
                      options={'maxiter': 600, 'rhobeg': 0.3})
    return result.fun / n


# Comparativa para n=6
n_cmp = 6
print(f'Comparativa para cadena de Heisenberg n={n_cmp}:')
print(f'  Referencia Bethe ansatz (n→∞): {E_bethe_per_site:.6f}')

# ED
E0_ed6, _ = exact_ground_energy(n_cmp)
print(f'  ED exacta:                     {E0_ed6/n_cmp:.6f}')

# iDMRG
ens6, _, _ = idmrg(n_sites=n_cmp + 2, chi=32)
E_dmrg6 = next((E for n, _, E in ens6 if n == n_cmp), ens6[-1][2])
print(f'  iDMRG (χ=32):                  {E_dmrg6:.6f}  error={abs(E_dmrg6 - E0_ed6/n_cmp):.2e}')

# VQE
print('  VQE (calculando...)  ', end='', flush=True)
E_vqe6 = vqe_heisenberg(n_cmp, n_layers=3)
print(f'\n  VQE (3 capas):                 {E_vqe6:.6f}  error={abs(E_vqe6 - E0_ed6/n_cmp):.2e}')

# Resumen gráfico
methods = ['ED exacta', 'iDMRG (χ=32)', 'VQE (3 capas)', 'Bethe ansatz']
energies_cmp = [E0_ed6/n_cmp, E_dmrg6, E_vqe6, E_bethe_per_site]
errors_cmp = [0, abs(E_dmrg6 - E0_ed6/n_cmp), abs(E_vqe6 - E0_ed6/n_cmp),
              abs(E_bethe_per_site - E0_ed6/n_cmp)]

fig, ax = plt.subplots(figsize=(8, 5))
colors = ['green', 'blue', 'orange', 'red']
bars = ax.bar(methods, energies_cmp, color=colors, alpha=0.8, edgecolor='white')
ax.axhline(E0_ed6/n_cmp, color='green', ls='--', alpha=0.5)
for bar, err in zip(bars, errors_cmp):
    if err > 1e-8:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                f'err={err:.1e}', ha='center', va='bottom', fontsize=8)
ax.set_ylabel('E₀/n', fontsize=11)
ax.set_title(f'Comparativa de métodos — Heisenberg n={n_cmp}', fontsize=12)
ax.grid(True, axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

print('\nConclusión: DMRG es prácticamente exacto para Heisenberg 1D.')
print('VQE puede competir pero requiere más capas para alta precisión.')
print('Para n grande (n>40), DMRG escala mejor que VQE en hardware NISQ.')

## Resumen

| Método | Coste por paso | Error típico | Escalado con n |
|--------|---------------|--------------|----------------|
| ED exacta | O(2^(3n)) | 0 | Exponencial — impráctica n>20 |
| iDMRG (χ=32) | O(n χ³) | ~10⁻⁵ | Polinomial en n |
| VQE (p=3) | O(shots × n) | ~10⁻² | Limitado por ruido + barren plateau |
| Bethe ansatz | O(n²) | 0 | Exacto solo para 1D integrables |

**La cadena de Heisenberg antiferromagnética 1D:**
- Es un sistema **cuántico crítico** (sin gap de energía, c=1 CFT)
- La correlación decae algebraicamente: $C(r) \sim (-1)^r/r$
- La entropía crece logarítmicamente: $S(l) \sim \frac{1}{3}\log l$ (área law logarítmica)
- DMRG lo resuelve con precisión arbitraria para cualquier n en 1D
- Es el benchmark estándar para algoritmos de simulación cuántica

**Ventaja cuántica:**
- Para cadenas 1D: DMRG gana siempre sobre VQE NISQ
- Para redes 2D (Hubbard 2D, Heisenberg 2D): DMRG se vuelve exponencial → aquí VQE puede tener ventaja en hardware tolerante a fallos